<a href="https://colab.research.google.com/github/kumarsirish/FDP-AGENENTIC-AI-RAG/blob/main/rag-ragas-03/fictional-department-rag-ragas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AGENTIC RAG - RAGAS

This notebook walks through building an Agentic RAG system using LangChain and FAISS. It also demonstrates how to evaluate the RAG pipeline using RAGAS, a framework for evaluating Retrieval-Augmented Generation systems across key metrics like faithfulness, answer relevancy, context precision, and context recall.

## What is RAGAS and Why is it Essential?

RAGAS (Retrieval-Augmented Generation As A Service) is a powerful framework specifically designed to evaluate the performance of Retrieval-Augmented Generation (RAG) systems. It goes beyond traditional metrics like BLEU or ROUGE by focusing on aspects critical to the success of RAG applications.

**Why RAGAS is essential for RAG systems:**

1.  **Addresses RAG-specific challenges:** RAG systems combine information retrieval with text generation. This introduces unique failure modes, such as hallucination (generating information not present in the source), irrelevance to the query, or poor context utilization. RAGAS is built to directly measure these issues.

2.  **Focus on factual accuracy and grounding:** RAGAS's `Faithfulness` metric directly checks if the claims made in the generated answer are supported by the retrieved context. This is crucial for applications where factual correctness is paramount.

3.  **Evaluates relevance:** The `Answer Relevancy` metric assesses whether the generated answer directly addresses the user's question, ensuring the system doesn't produce off-topic responses.

4.  **Assesses retrieval quality:** Unlike general-purpose NLP metrics, RAGAS includes `Context Precision` and `Context Recall`. These metrics evaluate the effectiveness of the retrieval component by checking if the retrieved documents are relevant and comprehensive enough to answer the question.

5.  **Human-like evaluation:** RAGAS uses an LLM (the "judge LLM") to evaluate answers and contexts. This allows for a more nuanced assessment that can understand semantics and context, closer to how a human would evaluate the system, rather than just relying on lexical overlap.

6.  **Actionable insights:** The individual metrics provide clear signals about where a RAG system might be failing. For example, a low `Faithfulness` score suggests issues with the generation model (e.g., hallucination), while low `Context Recall` points to problems with the retrieval mechanism (e.g., missing relevant documents). This helps developers pinpoint and address specific weaknesses.

7.  **Automation:** RAGAS automates the evaluation process, which can be time-consuming and subjective if done manually, especially for large datasets. This enables continuous integration and testing for RAG pipelines.

In essence, RAGAS provides a holistic and specialized approach to RAG evaluation, giving developers the tools to build more reliable, accurate, and relevant RAG applications.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## Why Traditional ML and NLP Metrics Fall Short for RAG Evaluation

While traditional machine learning (ML) and natural language processing (NLP) evaluation metrics are valuable in their respective domains, they are often insufficient or misleading when it comes to comprehensively assessing Retrieval-Augmented Generation (RAG) systems. Here's why:

### 1. Classical ML Metrics (Precision, Accuracy, Recall, F1-score)

These metrics are primarily designed for **classification or structured prediction tasks**, where there's a clear, single correct output label or value. They measure how well a model predicts a predefined category or number.

*   **Mismatch with Generative Tasks:** RAG systems are generative; they produce free-form text answers. There isn't a fixed set of classes or a single 'correct' string output to compare against for metrics like accuracy or precision in the classical sense.
*   **Lack of Nuance for Text:** These metrics cannot capture the linguistic nuances, fluency, coherence, or semantic meaning of generated text. A RAG system might generate a perfectly accurate answer in terms of content, but if it's poorly phrased or irrelevant to the user's intent, classical metrics wouldn't reflect this.
*   **No Concept of 'Grounding':** Crucially, classical ML metrics have no mechanism to check if the generated answer is *grounded* in the retrieved context, which is a core requirement for RAG to avoid hallucination.

### 2. BLEU, ROUGE, and BERTScore (for NLP tasks)

These metrics are designed for evaluating text generation tasks like machine translation, summarization, or dialogue systems by comparing the generated text to one or more reference texts. While they address some aspects of text generation, they have significant limitations for RAG:

#### BLEU (Bilingual Evaluation Understudy) and ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

*   **Lexical Overlap Focus:** Both BLEU and ROUGE primarily rely on n-gram overlap between the generated answer and a reference answer. They measure lexical similarity rather than semantic understanding.
*   **Ignores Semantic Meaning:** A RAG system might generate an answer that uses different phrasing but conveys the exact same correct information as the reference. BLEU/ROUGE would penalize this for lack of exact word match, failing to capture semantic equivalence.
*   **Poor for Factuality/Hallucination:** These metrics do not inherently check if the generated text is factually correct or if it's supported by the retrieved source documents. A generated answer could perfectly match a reference (scoring high), but if both contain hallucinations, BLEU/ROUGE wouldn't detect it.
*   **Multiple Valid Answers:** For many questions in RAG, there can be multiple equally valid ways to phrase an answer. Relying on a single reference makes these metrics less robust, as they don't account for paraphrasing or alternative correct formulations.
*   **Context Disregard:** They don't evaluate *how* the retrieved context was used or if it was sufficient and relevant. They only compare the final generated text to a reference.

#### BERTScore

*   **Semantic Similarity:** BERTScore improves upon BLEU/ROUGE by using contextual embeddings (like BERT) to measure semantic similarity between tokens in the generated and reference texts. This makes it better at handling paraphrasing than n-gram based metrics.
*   **Still Lacks Grounding Check:** While semantically aware, BERTScore still doesn't inherently verify if the generated answer is factually grounded in the *retrieved documents*. It compares the generated answer to a *reference answer*, not directly to the source context that the RAG system *should* have used.
*   **Reference Dependency:** Like BLEU/ROUGE, it still requires high-quality reference answers, which can be expensive and time-consuming to produce, especially for complex RAG tasks where answers can be long and varied.

### Why RAGAS is Better Suited

RAGAS directly addresses these limitations by introducing metrics like `Faithfulness`, `Answer Relevancy`, `Context Precision`, and `Context Recall`. These metrics are specifically designed to evaluate the unique challenges of RAG systems by:

*   **Checking grounding against retrieved context:** `Faithfulness` and `Context Recall` directly assess the relationship between the generated answer, the retrieved context, and the original query.
*   **Evaluating relevance to the query:** `Answer Relevancy` ensures the answer actually addresses the question asked.
*   **Assessing retrieval effectiveness:** `Context Precision` and `Context Recall` directly measure the quality of the retrieved information, which is a critical component of RAG.
*   **Using LLM-as-a-judge:** This approach allows for more nuanced and semantic evaluation than purely lexical or statistical methods, better reflecting human judgment.

## 1. Install Dependencies

In [ ]:
import os

if not os.path.exists('requirements.txt'):
    ! wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-ragas-03/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt -q


## 2. Set API Keys

In [ ]:
import os
from dotenv import load_dotenv
from google.colab import userdata

#From Google Colab
#from google.colab import userdata
#if not os.getenv("HF_TOKEN"):
#os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
#if not os.getenv("GEMINI_API_KEY"):
#os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
#except ImportError:
# Load from .env file (local development)
#load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

#For Kaggle
#user_secrets = UserSecretsClient()
#GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
#HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
#os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [ ]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2")
db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True,
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = db.similarity_search(query, k=2)

    # Verbose logging to see what the agent retrieves
    print(f"\n[Search Tool] Query: '{query}'")
    for i, doc in enumerate(results):
        print(f"[Search Tool] Result {i+1}: {doc.page_content[:100]}...")

    return "\n".join([doc.page_content for doc in results])
# ── Switch providers by setting MODEL_NAME — no code changes needed ──────────
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash   (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   huggingface  → MODEL_NAME=huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0
#   ollama       → MODEL_NAME=ollama:llama3
# ────────────────────────────────────────────────────────────────────────────

# Make loaded keys visible to init_chat_model via standard env-var names
#os.environ.setdefault("GOOGLE_API_KEY", GEMINI_API_KEY or "")

MODEL_NAME = "google_genai:gemini-2.5-flash"
MODEL_KEY= GEMINI_API_KEY
#MODEL_NAME = "huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"
#MODEL_KEY = HF_TOKEN

# MODEL_NAME = "openai:gpt-4o"

chat_llm = init_chat_model(MODEL_NAME, temperature=0.1)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "You are an intelligent Agentic RAG system. "
        "Decide whether document retrieval is needed. "
        "If needed, use the search_answers tool."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [ ]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # Get the last message content
    content = result["messages"][-1].content

    # Check if content is a list (as in your example) or a string
    if isinstance(content, list):
        # Join all text pieces found in the list
        clean_answer = " ".join([item.get("text", "") for item in content if isinstance(item, dict)])
    else:
        # If it's already a string, just use it
        clean_answer = str(content)

    print("\nAnswer:", clean_answer)


ask("What is DQE?")

In [ ]:
ask("How many students does DQE have?")

In [ ]:
ask("How many stuents does it have")

In [ ]:
ask("What postgraduate courses are offered in DQE?")

## 7. RAGAS


### What each metric tells you
| Metric | What it measures | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Is the answer grounded in the retrieved context? | No |
| **Answer Relevancy** | Is the answer relevant to the question? | No |
| **Context Precision** | Are the retrieved chunks actually useful? | Yes |
| **Context Recall** | Did retrieval capture what was needed to answer? | Yes |

Scores range from 0 to 1 — higher is better.

| Metric | Question it answers | Score guide |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved chunks? | < 0.5 → hallucination risk; > 0.8 → trustworthy |
| **Answer Relevancy** | Does the answer actually address the question asked? | < 0.5 → off-topic; > 0.8 → on-point |
| **Context Precision** | Are the top-ranked retrieved chunks the useful ones? | < 0.5 → noisy retrieval; > 0.8 → precise retrieval |
| **Context Recall** | Did retrieval surface everything needed to answer? | < 0.5 → missing evidence; > 0.8 → good coverage |

---

### What to do if scores are low

| Symptom | Likely cause | Fix |
|---|---|---|
| Low Faithfulness | LLM adds facts not in context | Use a stronger/instruction-tuned model; tighten the system prompt |
| Low Answer Relevancy | Model misunderstands the question | Improve the system prompt; switch to a better model |
| Low Context Precision | Wrong chunks retrieved | Tune `k`, try MMR retrieval, or re-embed with a better model |
| Low Context Recall | Needed chunk not retrieved | Increase `k`, reduce chunk size, or improve the embedding model |

---

> **Note on the judge LLM:** RAGAS uses the same `judge_llm` to score faithfulness and relevancy. If the judge is also TinyLlama (a small model), scores may be noisy or inconsistent — a stronger model (Gemini, GPT-4o) as the judge gives more reliable scores even when TinyLlama is the RAG agent.

In [ ]:
import sys, asyncio, pandas as pd
from types import ModuleType
from contextlib import asynccontextmanager

# asyncio.timeout() requires being inside a Task — not satisfied in Jupyter.
# Replace it with a no-op so RAGAS executor can run normally.
@asynccontextmanager
async def _timeout_noop(delay):
    yield
asyncio.timeout = _timeout_noop

for _mod_name, _attrs in {
    "langchain_community.chat_models.vertexai": ["ChatVertexAI"],
    "langchain_community.chat_models.google_palm": ["ChatGooglePalm"],
    "langchain_community.llms.vertexai": ["VertexAI"],
}.items():
    _mod = sys.modules.setdefault(_mod_name, ModuleType(_mod_name))
    for _attr in _attrs:
        if not hasattr(_mod, _attr):
            setattr(_mod, _attr, type(_attr, (), {}))

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

eval_questions = [
    "What is DQE?",
    "What courses are offered in DQE?",
    "What is the primary research focus of DQE?"
]

ground_truths = [
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems. DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "DQE offers 5 courses, including core subjects, electives, hands-on project work, a postgraduate module 'Foundations of Quantum AI', and an undergraduate elective 'Quantum Machine Learning'.",
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence. They investigate Quantum Neural Networks (QNNs) and Quantum Reinforcement Learning (QRL), and their flagship initiative is Project QuLearn."
]

def extract_answer(resp):
    content = resp["messages"][-1].content
    if isinstance(content, list):
        return " ".join(item.get("text", "") for item in content if isinstance(item, dict)).strip()
    return str(content).strip()

samples = [
    SingleTurnSample(
        user_input=q,
        response=extract_answer(agent.invoke({"messages": [{"role": "user", "content": q}]})),
        retrieved_contexts=[doc.page_content for doc in db.similarity_search(q, k=3)],
        reference=gt,
    )
    for q, gt in zip(eval_questions, ground_truths)
]

judge_llm = LangchainLLMWrapper(chat_llm)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

result = evaluate(
    dataset=EvaluationDataset(samples),
    metrics=[
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings),
        ContextPrecision(llm=judge_llm),
        ContextRecall(llm=judge_llm),
    ],
)

scores_df = result.to_pandas()[["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]]
scores_df.loc["mean"] = scores_df.mean(numeric_only=True)
scores_df.iloc[-1, 0] = "MEAN"
pd.set_option("display.float_format", "{:.3f}".format)
print(scores_df.to_string(index=False))